In [3]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install psycopg2-binary
!pip install sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 60.2 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import faiss

from collections import Counter

from sentence_transformers import (
SentenceTransformer
)

from sqlalchemy import (
create_engine
)

In [6]:
df = pd.read_csv(
    "cleaned_dataset.csv"
)

index = faiss.read_index(
    "capa_index.faiss"
)

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Retrieval assets loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval assets loaded


In [7]:
capa_df = pd.read_csv(
    "historical_capa_records.csv"
)

print(capa_df.shape)

capa_df.head()

(1000, 23)


,capa_id,open_date,close_date,days_to_close,product,department,severity,root_cause,regulatory_reference,incident_summary,...,effectiveness_check,effectiveness_rating,recurrence,investigator_name,investigator_role,approver_name,approver_role,ca_completion_date,pa_completion_date,ec_completion_date
0,CAPA-2022-0001,2022-09-15,2023-01-12,119,Amoxicillin 500mg Capsules,Packaging,High,Environmental Control Failure,21 CFR Part 211.130,Temperature excursion in cold room CR-06 — rea...,...,Zero temperature excursions exceeding 30 minut...,Effective,No,J. Patel,QA Manager,Dr. L. Fernandez,QP / Responsible Person,2022-09-27,2022-10-30,2023-01-05
1,CAPA-2020-0002,2020-12-19,2021-04-21,123,Morphine Sulfate 10mg/mL Injection,QC Laboratory,Medium,Documentation Error,USP <1058>,Batch record for batch B7924B missing authoris...,...,Zero missing signatures in EBR for 12 months. ...,Effective,No,R. Sharma,Engineering Lead,Dr. A. Gupta,Site Director,2021-01-18,2021-01-25,2021-04-11
2,CAPA-2019-0003,2019-02-18,2019-06-09,111,Warfarin 5mg Tablets,QC Laboratory,High,Process Deviation,21 CFR Part 211.192,Tablet hardness for batch A5803C failed specif...,...,Zero hardness OOS events on this press for 12 ...,Partially Effective,No,P. Singh,Production Manager,Dr. S. Mehta,QA Director,2019-03-07,2019-03-29,2019-05-27
3,CAPA-2018-0004,2018-08-18,2018-12-05,109,Heparin Sodium 5000IU/mL,Warehouse,Medium,Documentation Error,EU GMP Chapter 3,Superseded SOP SOP-048 v3.3 used during batch ...,...,Zero superseded SOP events for 12 months. Mont...,Effective,No,A. Kumar,QC Manager,Dr. L. Fernandez,QP / Responsible Person,2018-08-30,2018-10-26,2018-11-27
4,CAPA-2022-0005,2022-06-24,2022-09-16,84,Paracetamol 500mg Tablets,Packaging,High,Process Deviation,EU GMP Chapter 5,Dissolution failure at 30 minute timepoint for...,...,Zero dissolution failures for 12 months. 100% ...,Effective,No,P. Singh,Production Manager,Dr. L. Fernandez,QP / Responsible Person,2022-07-10,2022-07-29,2022-09-05


In [24]:
new_incident = (
"Unexpected HVAC failure caused environmental monitoring deviation."
)

In [25]:
query_embedding = model.encode(
    [new_incident]
)

In [26]:
k = 5

distances, indices = index.search(

    np.array(
        query_embedding
    ).astype(
        "float32"
    ),

    k

)

print(indices)

[[  85 1468 1538 1191  951]]


In [27]:
retrieved=[]

for i in indices[0]:

    row=df.iloc[i]

    retrieved.append({

        "incident":
        row["incident_description"],

        "root_cause":
        row["root_cause"],

        "severity":
        row["severity_level"]

    })

retrieved_df=pd.DataFrame(
retrieved
)

retrieved_df

,incident,root_cause,severity
0,HVAC system failure in cleanroom Room B202 res...,Environmental Control Failure,Medium
1,HVAC system failure in cleanroom Room B202 res...,Environmental Control Failure,High
2,HVAC system failure in cleanroom Room B202 res...,Environmental Control Failure,Low
3,HVAC system failure in cleanroom Room D104 res...,Environmental Control Failure,High
4,HVAC system failure in cleanroom Room A101 res...,Environmental Control Failure,High


In [28]:
from collections import Counter

root_cause = Counter(
    retrieved_df["root_cause"]
).most_common(1)[0][0]

severity = Counter(
    retrieved_df["severity"]
).most_common(1)[0][0]

print("Root Cause:", root_cause)
print("Severity:", severity)

Root Cause: Environmental Control Failure
Severity: High


In [29]:
recommendations = capa_df[

    (capa_df["root_cause"] == root_cause)

    &

    (capa_df["severity"] == severity)

]

print(
recommendations.shape
)

recommendations.head()

(81, 23)


,capa_id,open_date,close_date,days_to_close,product,department,severity,root_cause,regulatory_reference,incident_summary,...,effectiveness_check,effectiveness_rating,recurrence,investigator_name,investigator_role,approver_name,approver_role,ca_completion_date,pa_completion_date,ec_completion_date
0,CAPA-2022-0001,2022-09-15,2023-01-12,119,Amoxicillin 500mg Capsules,Packaging,High,Environmental Control Failure,21 CFR Part 211.130,Temperature excursion in cold room CR-06 — rea...,...,Zero temperature excursions exceeding 30 minut...,Effective,No,J. Patel,QA Manager,Dr. L. Fernandez,QP / Responsible Person,2022-09-27,2022-10-30,2023-01-05
14,CAPA-2020-0015,2020-10-05,2021-01-23,110,Vancomycin 500mg Powder for Infusion,Sterile Fill,High,Environmental Control Failure,21 CFR Part 211.113,Ambient temperature in dispensary A rose to 27...,...,Dispensary temperature within 15–25°C for 12 m...,Effective,No,A. Kumar,QC Manager,Dr. A. Gupta,Site Director,2020-10-28,2020-12-10,2021-01-11
33,CAPA-2019-0034,2019-03-19,2019-07-22,125,Morphine Sulfate 10mg/mL Injection,Manufacturing,High,Environmental Control Failure,EU GMP Chapter 5,Relative humidity in tablet manufacturing suit...,...,RH within 55% limit for 12 months on continuou...,Effective,No,R. Sharma,Engineering Lead,Dr. S. Mehta,QA Director,2019-04-11,2019-04-17,2019-07-15
44,CAPA-2023-0045,2023-04-04,2023-08-16,134,Amoxicillin 500mg Capsules,Sterile Fill,High,Environmental Control Failure,21 CFR Part 211.113,Relative humidity in tablet manufacturing suit...,...,RH within 55% limit for 12 months on continuou...,Effective,No,P. Singh,Production Manager,Dr. A. Gupta,Site Director,2023-05-01,2023-05-25,2023-08-07
46,CAPA-2019-0047,2019-01-09,2019-04-30,111,Enoxaparin 40mg/0.4mL Injection,Packaging,High,Environmental Control Failure,EU GMP Chapter 5,Temperature excursion in cold room CR-06 — rea...,...,Zero temperature excursions exceeding 30 minut...,Not Effective,Yes,M. Williams,QA Analyst,Dr. L. Fernandez,QP / Responsible Person,2019-01-29,2019-02-25,2019-04-22


In [30]:
corrective_actions = (

recommendations[
"corrective_action"

]

.dropna()

.tolist()

)

preventive_actions = (

recommendations[
"preventive_action"

]

.dropna()

.tolist()

)

In [31]:
corr_rank = Counter(
corrective_actions
)

prev_rank = Counter(
preventive_actions
)

In [32]:
print(
"\n===== CAPA RECOMMENDATIONS ====="
)

print(
"\nCorrective Actions:"
)

for action,count in (

corr_rank
.most_common(3)

):

    print(
f"- {action}"
)

print(
"\nPreventive Actions:"
)

for action,count in (

prev_rank
.most_common(3)

):

    print(
f"- {action}"
)


===== CAPA RECOMMENDATIONS =====

Corrective Actions:
- Filling halted. HEPA filter replaced. DOP test passed on replacement filter. Batch risk-assessed — sterility test initiated. Grade A re-qualified before restart.
- Dispensed materials quarantined and tested for conformance against stability data. AC unit repaired. Materials released after QA review of stability data.
- Fill area taken out of service. Full decontamination with sporicidal agent. Filter replaced. Grade A re-qualified over 3 consecutive days. Batch quarantined for sterility testing.

Preventive Actions:
- HEPA filter replacement scheduled at 10 months (not 12) to provide buffer. DOP test added to quarterly PM rather than annual. Filter age tracked in LIMS with auto-alert 60 days before due.
- Continuous temperature monitoring with alarm to QA office and engineering installed in all dispensary areas. AC unit added to monthly PM schedule. Backup portable AC unit available for emergency deployment.
- HVAC PM integrated 

Similar Historical Cases Count

In [33]:
historical_cases = len(
recommendations
)

print(
"Historical Similar Cases:",
historical_cases
)

Historical Similar Cases: 81


Recommendation Confidence

In [39]:
from collections import Counter
import numpy as np


# ----------------------------------
# 1. ROOT CAUSE
# ----------------------------------

root_cause = Counter(
    retrieved_df["root_cause"]
).most_common(1)[0][0]


# ----------------------------------
# 2. SEVERITY
# ----------------------------------

severity = Counter(
    retrieved_df["severity"]
).most_common(1)[0][0]


# ----------------------------------
# 3. CONFIDENCE SCORE
# ----------------------------------


top_distance = float(
distances[0][0]
)

confidence = float(

round(

100*np.exp(
-top_distance
/

(
float(
np.max(
distances
)
)

+1e-10

)

),

2

)

)

# ----------------------------------
# 4. SEARCH HISTORICAL CAPAs
# ----------------------------------

recommendations = capa_df[

(capa_df["root_cause"]==root_cause)

&

(capa_df["severity"]==severity)

]


# ----------------------------------
# 5. HISTORICAL CASE COUNT
# ----------------------------------

historical_cases = len(
recommendations
)


# ----------------------------------
# 6. EXTRACT ACTIONS
# ----------------------------------

corrective_actions = (

recommendations[
"corrective_action"

]

.dropna()

.tolist()

)

preventive_actions = (

recommendations[
"preventive_action"

]

.dropna()

.tolist()

)


# ----------------------------------
# 7. RANK ACTIONS
# ----------------------------------

corr_rank = Counter(
corrective_actions
)

prev_rank = Counter(
preventive_actions
)


# ----------------------------------
# 8. RECOMMENDATION REASON
# ----------------------------------

reason = f"""

Generated using:

• Similar historical incidents

• Root Cause:
{root_cause}

• Severity:
{severity}

• Historical CAPAs:
{historical_cases}

"""


# ----------------------------------
# FINAL OUTPUT
# ----------------------------------

print(
"\n===== CAPA RECOMMENDATION ====="
)

print(
f"""

Most Likely Root Cause:

{root_cause}

"""
)


print(
f"""

Estimated Severity:

{severity}

"""
)


print(
f"""

Confidence:

{confidence:2f}%

"""
)


print(
f"""

Historical Similar Cases:

{historical_cases}

"""
)


print(
"\nCorrective Actions:"
)

for action,count in (

corr_rank
.most_common(3)

):

    print(
f"- {action}"
)



print(
"\nPreventive Actions:"
)

for action,count in (

prev_rank
.most_common(3)

):

    print(
f"- {action}"
)



print(
"\nRecommendation Reason:"
)

print(
reason
)


===== CAPA RECOMMENDATION =====


Most Likely Root Cause:

Environmental Control Failure




Estimated Severity:

High




Confidence:

37.370000%




Historical Similar Cases:

81



Corrective Actions:
- Filling halted. HEPA filter replaced. DOP test passed on replacement filter. Batch risk-assessed — sterility test initiated. Grade A re-qualified before restart.
- Dispensed materials quarantined and tested for conformance against stability data. AC unit repaired. Materials released after QA review of stability data.
- Fill area taken out of service. Full decontamination with sporicidal agent. Filter replaced. Grade A re-qualified over 3 consecutive days. Batch quarantined for sterility testing.

Preventive Actions:
- HEPA filter replacement scheduled at 10 months (not 12) to provide buffer. DOP test added to quarterly PM rather than annual. Filter age tracked in LIMS with auto-alert 60 days before due.
- Continuous temperature monitoring with alarm to QA office and engineering inst

In [37]:
print(
"\n===== WHY THIS RECOMMENDATION ====="
)

print(
f"""

Top Similar Incident:

{retrieved_df.iloc[0]["incident"]}

"""
)

print(
f"""

Root Cause Match:

{root_cause}

"""
)

print(
f"""

Severity Match:

{severity}

"""
)

print(
f"""

Historical CAPAs Used:

{historical_cases}

"""
)

print(
f"""

Confidence:

{confidence}%

"""
)


===== WHY THIS RECOMMENDATION =====


Top Similar Incident:

HVAC system failure in cleanroom Room B202 resulted in differential pressure drop below specification. Grade integrity compromised for 6.3 hours during batch C9880A manufacture.




Root Cause Match:

Environmental Control Failure




Severity Match:

High




Historical CAPAs Used:

81




Confidence:

0%


